## KaggleのLLM-Classification Finetuningのためのベースライン

データの内部分析と前処理，モデル構築，訓練，評価を体系的に行っています

参照はこちら: https://www.kaggle.com/code/addisonhoward/lmsys-kerasnlp-starter

In [23]:
#import packages
import json
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm
import plotly.express as px

In [3]:
#define configuration
class CFG:
    seed = 42
    model_name = "microsoft/deberta-v3-small"
    max_length = 512
    batch_size = 8
    epochs = 3
    lr = 5e-6
    num_classes = 3
    label2name = {0: 'winner_model_a', 1: 'winner_model_b', 2: 'winner_tie'}
    name2label = {v: k for k, v in label2name.items()}

In [4]:
#define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
#define base path
BASE_PATH = "data/llm-classification-finetuning"

## 1. データ分析と前処理

データをロードし，プロンプトやレスポンスがnullだった場合，何も入力しないようにします "null" → ""

In [6]:
#Load data and convert "null" to ""
df = pd.read_csv(f"{BASE_PATH}/train.csv")
df["prompt"] = df["prompt"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")
df["response_a"] = df["response_a"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")
df["response_b"] = df["response_b"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")

test_df = pd.read_csv(f"{BASE_PATH}/test.csv")
test_df["prompt"] = test_df["prompt"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")
test_df["response_a"] = test_df["response_a"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")
test_df["response_b"] = test_df["response_b"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")

df["class_name"] = df[["winner_model_a", "winner_model_b", "winner_tie"]].idxmax(axis=1)
df["label"] = df["class_name"].map(CFG.name2label)

In [7]:
#concat prompt and response
def build_text(prompt, response):
    return f"Prompt: {prompt}\n\nResponse: {response}"

df["text_a"] = df.apply(lambda r: build_text(r["prompt"], r["response_a"]), axis=1)
df["text_b"] = df.apply(lambda r: build_text(r["prompt"], r["response_b"]), axis=1)
test_df["text_a"] = test_df.apply(lambda r: build_text(r["prompt"], r["response_a"]), axis=1)
test_df["text_b"] = test_df.apply(lambda r: build_text(r["prompt"], r["response_b"]), axis=1)

In [ ]:
display(df.head(2))
display(test_df.head())

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie,class_name,label,text_a,text_b
0,30192,gpt-4-1106-preview,gpt-4-0613,Is it morally right to try to have a certain p...,The question of whether it is morally right to...,"As an AI, I don't have personal beliefs or opi...",1,0,0,winner_model_a,0,Prompt: Is it morally right to try to have a c...,Prompt: Is it morally right to try to have a c...
1,53567,koala-13b,gpt-4-0613,What is the difference between marriage licens...,A marriage license is a legal document that al...,A marriage license and a marriage certificate ...,0,1,0,winner_model_b,1,Prompt: What is the difference between marriag...,Prompt: What is the difference between marriag...
2,65089,gpt-3.5-turbo-0613,mistral-medium,explain function calling. how would you call a...,Function calling is the process of invoking or...,Function calling is the process of invoking a ...,0,0,1,winner_tie,2,Prompt: explain function calling. how would yo...,Prompt: explain function calling. how would yo...
3,96401,llama-2-13b-chat,mistral-7b-instruct,How can I create a test set for a very rare ca...,Creating a test set for a very rare category c...,When building a classifier for a very rare cat...,1,0,0,winner_model_a,0,Prompt: How can I create a test set for a very...,Prompt: How can I create a test set for a very...
4,198779,koala-13b,gpt-3.5-turbo-0314,What is the best way to travel from Tel-Aviv t...,The best way to travel from Tel Aviv to Jerusa...,The best way to travel from Tel-Aviv to Jerusa...,0,1,0,winner_model_b,1,Prompt: What is the best way to travel from Te...,Prompt: What is the best way to travel from Te...


,id,prompt,response_a,response_b,text_a,text_b
0,136060,"I have three oranges today, I ate an orange ye...",You have two oranges today.,You still have three oranges. Eating an orange...,"Prompt: I have three oranges today, I ate an o...","Prompt: I have three oranges today, I ate an o..."
1,211333,You are a mediator in a heated political debat...,Thank you for sharing the details of the situa...,Mr Reddy and Ms Blue both have valid points in...,Prompt: You are a mediator in a heated politic...,Prompt: You are a mediator in a heated politic...
2,1233961,How to initialize the classification head when...,When you want to initialize the classification...,To initialize the classification head when per...,Prompt: How to initialize the classification h...,Prompt: How to initialize the classification h...


プロンプトとレスポンスの結合 "Prompt: 'prompt' \n\n Response: 'response'"

結合したもの df["option"] 入力とする．これはNLPの"multiple-choice question task"から流用

In [9]:
# Define a function to create options based on the prompt and choices
def make_pairs(row):
    row["encode_fail"] = False
    try:
        prompt = row.prompt.encode("utf-8").decode("utf-8")
    except:
        prompt = ""
        row["encode_fail"] = True

    try:
        response_a = row.response_a.encode("utf-8").decode("utf-8")
    except:
        response_a = ""
        row["encode_fail"] = True

    try:
        response_b = row.response_b.encode("utf-8").decode("utf-8")
    except:
        response_b = ""
        row["encode_fail"] = True
        
    row['options'] = [f"Prompt: {prompt}\n\nResponse: {response_a}",  # Response from Model A
                      f"Prompt: {prompt}\n\nResponse: {response_b}"  # Response from Model B
                      ]
    return row

In [10]:
df = df.apply(make_pairs, axis=1)  # Apply the make_pairs function to each row in df
display(df.head(2))  # Display the first 2 rows of df

test_df = test_df.apply(make_pairs, axis=1)  # Apply the make_pairs function to each row in df
display(test_df.head(2))  # Display the first 2 rows of df

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie,class_name,label,text_a,text_b,encode_fail,options
0,30192,gpt-4-1106-preview,gpt-4-0613,Is it morally right to try to have a certain p...,The question of whether it is morally right to...,"As an AI, I don't have personal beliefs or opi...",1,0,0,winner_model_a,0,Prompt: Is it morally right to try to have a c...,Prompt: Is it morally right to try to have a c...,False,[Prompt: Is it morally right to try to have a ...
1,53567,koala-13b,gpt-4-0613,What is the difference between marriage licens...,A marriage license is a legal document that al...,A marriage license and a marriage certificate ...,0,1,0,winner_model_b,1,Prompt: What is the difference between marriag...,Prompt: What is the difference between marriag...,False,[Prompt: What is the difference between marria...


,id,prompt,response_a,response_b,text_a,text_b,encode_fail,options
0,136060,"I have three oranges today, I ate an orange ye...",You have two oranges today.,You still have three oranges. Eating an orange...,"Prompt: I have three oranges today, I ate an o...","Prompt: I have three oranges today, I ate an o...",False,"[Prompt: I have three oranges today, I ate an ..."
1,211333,You are a mediator in a heated political debat...,Thank you for sharing the details of the situa...,Mr Reddy and Ms Blue both have valid points in...,Prompt: You are a mediator in a heated politic...,Prompt: You are a mediator in a heated politic...,False,[Prompt: You are a mediator in a heated politi...


In [ ]:
# count the number of encoding failures
df.encode_fail.value_counts(normalize=False)

encode_fail
False    57439
True        38
Name: count, dtype: int64

データセット内で使用されているLLMの分布の可視化

In [ ]:
# Visualize the distributuion of LLMs in the dataset
model_df = pd.concat([df.model_a, df.model_b])
counts = model_df.value_counts().reset_index()
counts.columns = ['LLM', 'Count']

# Create a bar plot with custom styling using Plotly
fig = px.bar(counts, x='LLM', y='Count',
             title='Distribution of LLMs',
             color='Count', color_continuous_scale='viridis')

fig.update_layout(xaxis_tickangle=-45)  # Rotate x-axis labels for better readability

fig.show()

A, Bのどちらが好ましい回答だったかを分布で可視化

A, Bの分布に偏りがないことを示す

In [ ]:
# Visualize the distribution of winners in the dataset
counts = df['class_name'].value_counts().reset_index()
counts.columns = ['Winner', 'Win Count']

fig = px.bar(counts, x='Winner', y='Win Count',
             title='Winner distribution for Train Data',
             labels={'Winner': 'Winner', 'Win Count': 'Win Count'},
             color='Winner', color_continuous_scale='viridis')

fig.update_layout(xaxis_title="Winner", yaxis_title="Win Count")

fig.show()

## 2. モデル構築

In [ ]:
# Split the data into training and validation sets with startified sampling to maintain class distribution
train_df, valid_df = train_test_split(df, test_size=0.2, random_state=CFG.seed, stratify=df["label"])

In [ ]:
# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

モデルに入力するためのカスタムデータセットの作成



In [ ]:
# Define a custom dataset class for the LLM classification task
# LMSYS is the name of the organization that hosts a Chatbot Arena
class LMSYSDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        enc_a = self.tokenizer(
            row["text_a"],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        enc_b = self.tokenizer(
            row["text_b"],
            truncation=True,
            max_length = self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        item = {
            "input_ids_a": enc_a["input_ids"].squeeze(0),
            "attention_mask_a": enc_a["attention_mask"].squeeze(0),
            "input_ids_b": enc_b["input_ids"].squeeze(0),
            "attention_mask_b": enc_b["attention_mask"].squeeze(0)
        }

        if not self.is_test:
            item["label"] = torch.tensor(row["label"], dtype=torch.long)

        return item

In [17]:
train_ds = LMSYSDataset(train_df, tokenizer, CFG.max_length, is_test=False)
valid_ds = LMSYSDataset(valid_df, tokenizer, CFG.max_length, is_test=False)
test_ds = LMSYSDataset(test_df, tokenizer, CFG.max_length, is_test=True)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=2)

分類モデルの構築

入力: (prompt + response_a) & (prompt + response_b)

出力: 3次元の確率分布 (winner_a, winner_b, winner_tie)

Deberta → (batch, token, embedding) → pooling → (batch, embedding) → classifier → (batch, 3)

In [ ]:
# Define the pairwise DeBERTa classifier model
class PairwiseDebertaClassifier(nn.Module):
    def __init__(self, model_name, num_classes=3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size*2, num_classes)

    def mean_pool(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return summed / denom
    
    def encode(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pool(outputs.last_hidden_state, attention_mask)
        return pooled
    
    def forward(self, input_ids_a, attention_mask_a, input_ids_b, attention_mask_b):
        feat_a = self.encode(input_ids_a, attention_mask_a)
        feat_b = self.encode(input_ids_b, attention_mask_b)
        feats = torch.cat([feat_a, feat_b], dim=1)
        logits = self.classifier(feats)
        return logits

In [25]:
model = PairwiseDebertaClassifier(CFG.model_name, CFG.num_classes).to(device)

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

c:\Users\kajita\Desktop\llm-finetune\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kajita\.cache\huggingface\hub\models--microsoft--deberta-v3-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

In [26]:
#train setup
criterion = nn.CrossEntropyLoss(label_smoothing=0.02)
optimizer = AdamW(model.parameters(), lr=CFG.lr)

num_training_steps = len(train_loader) * CFG.epochs
num_warmup_steps = int(0.1 * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

In [27]:
#Train loop
def run_epoch(loader, model, criterion, optimizer=None, scheduler=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0
    total_correct = 0
    total_count = 0
    
    for batch in tqdm(loader):
        input_ids_a = batch["input_ids_a"].to(device)
        attention_mask_a = batch["attention_mask_a"].to(device)
        input_ids_b = batch["input_ids_b"].to(device)
        attention_mask_b = batch["attention_mask_b"].to(device)
        labels = batch["labels"].to(device)
        
        with torch.set_grad_enable(is_train):
            logits = model(
                input_ids_a = input_ids_a,
                attention_mask_a = attention_mask_a,
                input_ids_b = input_ids_b,
                attention_mask_b = attention_mask_b
            )
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()
        
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_count += labels.size(0)

    return total_loss / total_count, total_correct / total_count

In [ ]:
best_val_loss = float("inf")

for epoch in range(CFG.epochs):
    train_loss, train_acc = run_epoch(train_loader, model, criterion, optimizer, scheduler)
    val_loss, val_acc = run_epoch(valid_loader, model, criterion)

    print(f"Epoch {epoch+1}")
    print(f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}")
    print(f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        torch.save(model.state_dict(), "best_model.pth")
        best_val_loss = val_loss

  0%|          | 0/5748 [00:00<?, ?it/s]

In [ ]:
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()
all_probs = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids_a = batch["input_ids_a"].to(device)
        attention_mask_a = batch["attention_mask_a"].to(device)
        input_ids_b = batch["input_ids_b"].to(device)
        attention_mask_b = batch["attention_mask_b"].to(device)

        logits = model(
            input_ids_a = input_ids_a,
            attention_mask_a = attention_mask_a,
            input_ids_b = input_ids_b,
            attention_mask_b = attention_mask_b
        )
        probs = torch.softmax(logits, dim=1)
        all_probs.append(probs.cpu())
    
all_probs = torch.cat(all_probs, dim=0).numpy()

In [ ]:
sub_df = test_df[["id"]].copy()
sub_df["winner_model_a"] = all_probs[:, 0]
sub_df["winner_model_b"] = all_probs[:, 1]
sub_df["winner_tie"] = all_probs[:, 2]
sub_df.to_csv("submission.csv", index=False)